# Mat & Comp – ML 3: Descenso de gradiente
## Bajar la montaña con los ojos vendados

**Curso:** Machine learning y redes neuronales (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 2

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar la idea del **descenso de gradiente**: dar pasos en la dirección en que el error baja.
2. Calcular la **pendiente** (derivada) de una función de forma aproximada con numpy.
3. Elegir una **tasa de aprendizaje** y reconocer cuándo es demasiado pequeña o demasiado grande.
4. **Entrenar** la regresión lineal de ML 2 con descenso de gradiente, desde cero.
5. Entender por qué este método es el corazón del entrenamiento de las **redes neuronales**.

## 🔍 Perdido en la niebla

Estás en una montaña con una niebla tan espesa que no ves nada, ni siquiera tus pies. Quieres llegar al punto **más bajo del valle**, donde está el refugio.

No tienes mapa ni GPS. Lo único que puedes hacer es **sentir con los pies** si el suelo se inclina hacia un lado o hacia el otro.

✏️ *Escribe una estrategia para llegar al fondo del valle. ¿Qué harías si das un paso y el suelo empieza a subir? ¿Darías pasos grandes o pequeños?*

**Mi estrategia:** 

En ML 2 vimos el **paisaje del error** de la regresión lineal: un valle cuyo fondo es la mejor recta. Allí lo pintamos completo probando miles de puntos. Hoy vamos a encontrar el fondo **sin ver el mapa**, como en la niebla.

## 1. Una montaña de una sola dimensión

Empecemos con un valle sencillo: la función

$$f(x) = (x - 3)^2 + 1$$

Ya sabemos que su mínimo está en x = 3 (donde vale 1), pero hagamos de cuenta que no lo sabemos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def f(x):
    return (x - 3) ** 2 + 1

xs = np.linspace(-2, 8, 200)
plt.plot(xs, f(xs))
plt.scatter([0], [f(0)], color="red", s=80, zorder=3, label="empezamos aquí: x = 0")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Un valle de una dimensión")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 2. Sentir la inclinación: la pendiente

La **pendiente** de una curva en un punto dice hacia dónde sube y qué tan empinada es:
- pendiente **positiva**: la curva sube hacia la derecha, así que el valle está a la **izquierda**;
- pendiente **negativa**: la curva baja hacia la derecha, así que el valle está a la **derecha**;
- pendiente **cero**: estamos en un punto plano, ¡quizá en el fondo!

Podemos medirla sin cálculo, dando un pasito diminuto `h` y viendo cuánto cambia `f`:

$$\text{pendiente} \approx \frac{f(x + h) - f(x)}{h}$$

En cálculo esto se llama la **derivada** de `f`. Para varias variables, el conjunto de pendientes se llama **gradiente**.

In [ ]:
def pendiente(f, x, h=0.0001):
    return (f(x + h) - f(x)) / h

for punto in [0, 2, 3, 5]:
    print(f"En x = {punto}, la pendiente es aproximadamente {pendiente(f, punto):.3f}")

💡 Para esta función, el cálculo dice que la derivada exacta es 2(x − 3): da −6, −2, 0 y 4 en esos puntos. ¡La aproximación es casi perfecta!

## 3. El algoritmo: un paso cuesta abajo

La regla del **descenso de gradiente** es:

$$x_\text{nuevo} = x - \alpha \cdot \text{pendiente}(x)$$

- El signo **menos** hace que siempre caminemos en contra de la subida, es decir, **cuesta abajo**.
- **α** (alfa) es la **tasa de aprendizaje**: el tamaño de los pasos.
- Si la pendiente es grande (terreno empinado), el paso es grande; cerca del fondo, la pendiente es pequeña y los pasos se acortan solos.

In [ ]:
x = 0.0             # punto de partida
alfa = 0.1          # tasa de aprendizaje
historia = [x]

for paso in range(20):
    x = x - alfa * pendiente(f, x)
    historia.append(x)

print("Primeros pasos:", np.round(historia[:6], 3))
print(f"Después de 20 pasos: x = {x:.4f}, f(x) = {f(x):.4f}")

In [ ]:
historia = np.array(historia)
plt.plot(xs, f(xs), color="lightgray")
plt.plot(historia, f(historia), "o-", color="red", markersize=5)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("El camino del descenso de gradiente")
plt.grid(alpha=0.3)
plt.show()

Fíjate cómo los pasos son largos al principio y cada vez más cortos al acercarse al fondo.

## 4. 🎛️ La tasa de aprendizaje

Elegir α es un arte:
- si es **muy pequeña**, el algoritmo avanza lentísimo;
- si es **muy grande**, se pasa del fondo, rebota de un lado al otro… ¡y puede salir disparado!

Mueve los sliders y encuentra los tres comportamientos.

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def explorar_alfa(alfa=0.1, pasos=15, inicio=0.0):
    x = inicio
    camino = [x]
    for _ in range(pasos):
        x = x - alfa * pendiente(f, x)
        camino.append(x)
        if abs(x) > 1e6:
            break
    camino = np.array(camino)
    plt.figure(figsize=(7, 4.5))
    plt.plot(xs, f(xs), color="lightgray")
    visibles = np.abs(camino - 3) < 6
    plt.plot(camino[visibles], f(camino[visibles]), "o-", color="red", markersize=5)
    plt.ylim(0, 30)
    plt.xlim(-2, 8)
    estado = "¡DIVERGE!" if abs(camino[-1] - 3) > abs(inicio - 3) else f"x final = {camino[-1]:.3f}"
    plt.title(f"α = {alfa:.2f}, {len(camino) - 1} pasos  →  {estado}")
    plt.grid(alpha=0.3)
    plt.show()

interact(explorar_alfa,
         alfa=FloatSlider(value=0.1, min=0.01, max=1.1, step=0.01, continuous_update=False),
         pasos=IntSlider(value=15, min=1, max=60, continuous_update=False),
         inicio=FloatSlider(value=0.0, min=-2, max=8, step=0.5, continuous_update=False));

**Pistas para explorar:** prueba α = 0.01, 0.1, 0.5, 0.9, 1.0 y 1.05. ¿Con cuál llega exactamente al fondo en un solo paso? ¿Con cuál rebota? ¿Con cuál se escapa?

## 5. Ahora con dos perillas: la regresión lineal

Volvamos a los 60 estudiantes de ML 1 y ML 2. Queremos la mejor recta `nota = m·horas + b`, es decir, el punto `(m, b)` con el menor **error cuadrático medio**:

$$\text{ECM}(m, b) = \frac{1}{n}\sum_{i=1}^{n} \left(y_i - (m x_i + b)\right)^2$$

Ahora el "terreno" tiene dos direcciones, así que necesitamos **dos pendientes**: una para `m` y otra para `b`. Juntas forman el **gradiente**. Con cálculo se obtiene:

$$\frac{\partial\, \text{ECM}}{\partial m} = -\frac{2}{n}\sum x_i \,(y_i - (m x_i + b)) \qquad \frac{\partial\, \text{ECM}}{\partial b} = -\frac{2}{n}\sum (y_i - (m x_i + b))$$

Con numpy, cada fórmula es una sola línea.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
df = pd.read_csv(url)
x = df["horas_estudio"].to_numpy()
y = df["nota_matematicas"].to_numpy()

def ecm(m, b):
    return np.mean((y - (m * x + b)) ** 2)

def gradiente(m, b):
    error = y - (m * x + b)
    d_m = -2 * np.mean(x * error)
    d_b = -2 * np.mean(error)
    return d_m, d_b

print("ECM empezando en m = 0, b = 0:", round(ecm(0, 0), 3))
print("Gradiente en ese punto:", np.round(gradiente(0, 0), 3))

Las dos pendientes son negativas: el error **baja** si aumentamos `m` y `b`. ¡Eso es lo que el algoritmo hará!

### El ciclo de entrenamiento

Este es, en miniatura, el mismo ciclo con el que se entrenan las redes neuronales más grandes del mundo:
1. calcular el gradiente del error;
2. mover cada parámetro un paso en contra de su pendiente;
3. repetir.

In [ ]:
m, b = 0.0, 0.0
alfa = 0.01
historia_m, historia_b, historia_ecm = [m], [b], [ecm(m, b)]

for paso in range(2000):
    d_m, d_b = gradiente(m, b)
    m = m - alfa * d_m
    b = b - alfa * d_b
    historia_m.append(m)
    historia_b.append(b)
    historia_ecm.append(ecm(m, b))
    if paso in [0, 9, 99, 499, 1999]:
        print(f"paso {paso + 1:5d}:  m = {m:.4f}   b = {b:.4f}   ECM = {ecm(m, b):.4f}")

In [ ]:
# Comparamos con la fórmula exacta de mínimos cuadrados (ML 2)
m_exacta = np.sum((x - x.mean()) * (y - y.mean())) / np.sum((x - x.mean()) ** 2)
b_exacta = y.mean() - m_exacta * x.mean()
print(f"Descenso de gradiente: m = {m:.4f}, b = {b:.4f}")
print(f"Mínimos cuadrados:     m = {m_exacta:.4f}, b = {b_exacta:.4f}")

¡Llegamos al mismo resultado, **sin usar la fórmula**! Solo con pasos cuesta abajo.

## 6. Ver el aprendizaje

### La curva de aprendizaje

Graficar el error en cada paso es lo primero que se mira cuando se entrena un modelo. Si baja y se aplana, el entrenamiento va bien.

In [ ]:
plt.plot(historia_ecm)
plt.yscale("log")
plt.xlabel("paso")
plt.ylabel("ECM (escala logarítmica)")
plt.title("Curva de aprendizaje")
plt.grid(alpha=0.3)
plt.show()

### El camino en el paisaje del error

Este es el mismo mapa de ML 2. La línea roja es el camino que siguió el algoritmo desde (0, 0) hasta el fondo del valle.

In [ ]:
valores_m = np.linspace(-0.1, 0.6, 150)
valores_b = np.linspace(-0.3, 2.5, 150)
M, B = np.meshgrid(valores_m, valores_b)
mapa = np.mean((y[:, None, None] - (M * x[:, None, None] + B)) ** 2, axis=0)

plt.figure(figsize=(7, 5))
plt.contourf(M, B, np.log(mapa), levels=30, cmap="viridis")
plt.colorbar(label="log(ECM)")
plt.plot(historia_m, historia_b, color="red", linewidth=2, label="camino del descenso")
plt.scatter([historia_m[0]], [historia_b[0]], color="white", s=60, zorder=3, label="inicio")
plt.scatter([m_exacta], [b_exacta], color="red", marker="*", s=200, zorder=3, label="mínimo exacto")
plt.xlabel("m (pendiente)")
plt.ylabel("b (intercepto)")
plt.title("El descenso de gradiente en el paisaje del error")
plt.legend(loc="upper right")
plt.show()

¿Ves la forma del camino? Primero baja rápido hacia el "canal" del valle y luego avanza despacio por el fondo, que es casi plano. Por eso las primeras decenas de pasos mejoran mucho y los últimos cientos solo afinan.

### La recta aprendiendo

In [ ]:
plt.scatter(x, y, alpha=0.5, color="gray")
lineas = np.linspace(0, 14, 50)
for paso, color in zip([0, 5, 20, 100, 2000], ["#fde725", "#5ec962", "#21918c", "#3b528b", "#440154"]):
    plt.plot(lineas, historia_m[paso] * lineas + historia_b[paso], color=color, label=f"paso {paso}")
plt.xlabel("horas de estudio")
plt.ylabel("nota")
plt.title("La recta mejora en cada paso")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. ⚠️ Cuando la tasa de aprendizaje es muy grande

Con `alfa = 0.01` todo salió bien. ¿Qué pasa con un valor apenas un poco mayor?

In [ ]:
for alfa in [0.001, 0.01, 0.015, 0.016]:
    m, b = 0.0, 0.0
    for paso in range(1000):
        d_m, d_b = gradiente(m, b)
        m, b = m - alfa * d_m, b - alfa * d_b
    print(f"alfa = {alfa:<6}  →  después de 1000 pasos: m = {m:.4g}, ECM = {ecm(m, b):.4g}")

Con α = 0.001 el algoritmo va bien pero lento (todavía no llega). Con 0.01 y 0.015 converge. ¡Y con 0.016 el error **explota** hasta números astronómicos! Cada paso se pasa tanto del fondo que termina más arriba que antes, y así cada vez peor.

En la práctica, quienes entrenan redes neuronales prueban varias tasas de aprendizaje y miran la curva de aprendizaje para elegir la mejor.

## 8. ¿Para qué todo esto, si ya teníamos la fórmula?

Para la regresión lineal existe una fórmula exacta, pero es una excepción. En casi todos los modelos interesantes, **no hay fórmula**:

| Modelo | Parámetros | ¿Fórmula exacta? |
|---|---|---|
| Recta de ML 2 | 2 | sí |
| Una red neuronal pequeña para reconocer dígitos | ~100 000 | no |
| Un modelo de lenguaje grande, como los asistentes de IA | miles de millones | no |

El descenso de gradiente funciona igual con 2 perillas que con miles de millones: calcular el gradiente, dar un paso cuesta abajo, repetir. En las lecciones de redes neuronales (ML 8 a ML 12) vas a usar exactamente este ciclo.

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …empiezas el descenso de la regresión en `m = 1, b = 5` en vez de `(0, 0)`? ¿Llega al mismo lugar?
2. …en la función `f(x)` usas `alfa = 0.5`? ¿Cuántos pasos necesita? (Pista: piensa en la pendiente exacta 2(x − 3).)
3. …cambias el signo del paso a `x = x + alfa * pendiente(f, x)`? ¿Hacia dónde va?
4. …usas una función con **dos valles**, como `g(x) = x**4 - 4*x**2 + x`, y empiezas en x = 2 o en x = −2? ¿Llegas siempre al valle más profundo?
5. …usas `h = 1` en la función `pendiente`? ¿Qué tan buena es la aproximación?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil).** Usa descenso de gradiente para encontrar el mínimo de `p(x) = x**2 - 6*x + 10`, empezando en x = 0 con α = 0.1 y 50 pasos. ¿En qué x queda y cuánto vale `p` allí? Compruébalo completando el cuadrado: x² − 6x + 10 = (x − 3)² + 1.

**Ejercicio 2 (fácil). La derivada aproximada.** Para `c(x) = x**3`, calcula la pendiente aproximada en x = 2 con `h = 0.1`, `h = 0.01` y `h = 0.0001`. La respuesta exacta es 12. ¿Cómo cambia el error al hacer `h` más pequeño?

**Ejercicio 3 (medio).** Entrena con descenso de gradiente la recta que predice la **nota** a partir de las **horas de redes** (`horas_redes`). Elige α y el número de pasos, grafica la curva de aprendizaje y compara el resultado con `np.polyfit(x, y, 1)`.

**Ejercicio 4 (medio). Dos dimensiones.** Minimiza la función `q(a, c) = (a - 1)**2 + 2*(c + 2)**2` empezando en (a, c) = (4, 3). Sus pendientes son `2*(a - 1)` y `4*(c + 2)`. Guarda el camino y dibújalo sobre un mapa de contorno (como en la sección 6). ¿A qué punto llega?

**Ejercicio 5 (medio). Normalizar.** El valle de la regresión es largo y estrecho porque `x` llega hasta 14. Crea `x_norm = (x - x.mean()) / x.std()` y repite el descenso de gradiente con `x_norm`. ¿Qué tasa de aprendizaje máxima soporta ahora sin explotar? ¿Cuántos pasos necesita para llegar a un ECM de 0.1232?

## 🏆 Reto opcional: descenso de gradiente estocástico

Con millones de datos, calcular el gradiente con **todos** en cada paso es muy lento. El **descenso de gradiente estocástico** (SGD) usa en cada paso **solo un estudiante elegido al azar**:

```python
i = np.random.randint(len(x))
error = y[i] - (m * x[i] + b)
d_m = -2 * x[i] * error
d_b = -2 * error
```

1. Entrena la recta así durante 5000 pasos con α = 0.002 y grafica la curva de aprendizaje (el ECM con todos los datos, calculado cada 10 pasos). ¿Cómo se ve comparada con la de la sección 6?
2. ¿Llega exactamente al mínimo o se queda "temblando" cerca? ¿Qué pasa si reduces α poco a poco durante el entrenamiento?

## 🎬 Cierre

Hoy aprendiste que:
- El **descenso de gradiente** busca el mínimo de una función dando pasos en contra de la pendiente: `x ← x − α · pendiente`.
- La pendiente se puede aproximar con `(f(x + h) − f(x)) / h`; con varias variables, las pendientes forman el **gradiente**.
- La **tasa de aprendizaje** α controla el tamaño de los pasos: muy pequeña es lenta y muy grande hace que el error explote.
- Con este método entrenamos la regresión lineal **sin fórmula** y llegamos al mismo resultado que con mínimos cuadrados.
- La **curva de aprendizaje** muestra si el entrenamiento va bien.

**Para pensar:** hasta ahora hemos predicho **números** (la nota). Pero muchos problemas piden predecir **categorías**: ¿este correo es spam o no?, ¿esta foto es de un perro o de un gato?, ¿este estudiante aprobará o no? En la próxima lección veremos un método muy intuitivo para clasificar: preguntarles a los **vecinos más cercanos**. 🚀